# 09 · V-JEPA 2.1-B continuous CAN v4-A — ordinal → scalar residual fusion

v3-A established that the frozen V-JEPA + BiGRU representation contains useful longitudinal information: the ordinal branch separates ACCEL/DECEL events well, while the scalar acceleration output still underestimates magnitude and sends many dynamic frames to CONSTANT.

v4-A is a **continuation/fusion experiment**. It warm-starts from the completed v3-A `best.pt`, keeps the V-JEPA backbone frozen, preserves all v2/v3 continuous-CAN and ordinal losses, and adds only a small residual fusion module that explicitly feeds the ordinal representation into the scalar acceleration prediction.

The fusion input contains:

- the existing raw scalar acceleration prediction,
- all `P(a < -threshold)` and `P(a > +threshold)` ordinal probabilities,
- a signed ordinal motion score,
- a threshold-weighted signed magnitude score,
- an ordinal activity score.

The final prediction is `raw_accel + gate * residual_mlp(...)`. The residual MLP's last layer is initialized to zero, so immediately after loading v3-A the fused output is **exactly identical** to the v3-A raw scalar output. This makes the warm-start safe and isolates the effect of learning the new fusion path.

Ordinal probabilities are detached before entering the fusion MLP. Scalar-regression gradients therefore cannot corrupt the already-useful ordinal branch; that branch continues to train only through its balanced ordinal BCE/monotonic objectives.

**Official-metric safety:** `stage3/metrics.py` is not modified. The comma2k19 ordinal thresholds remain training/diagnostic values only and are not claimed to be DACON's hidden class thresholds.

**Resume policy:** a fresh v4-A run loads only v3-A `best.pt` model weights with the newly added fusion parameters initialized separately. After v4-A has started, only its own `latest.pt` is used for resume; the v3-A optimizer/scheduler is never resumed.


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    message=r".*torch\.backends\.cuda\.sdp_kernel\(\).*deprecated.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*Importing from timm\.models\.layers is deprecated.*",
    category=FutureWarning,
)

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Initial Drive mount failed; forcing remount:", repr(exc))
    drive.mount("/content/drive", force_remount=True)

try:
    _ = next(Path("/content/drive/MyDrive").iterdir(), None)
except OSError as exc:
    print("Drive mount is stale; forcing remount:", repr(exc))
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current_branch = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current_branch != BRANCH:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)

    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        print("WARNING: local repo has changes; git pull skipped.")
    else:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )

# Keep Colab's binary stack; install only required extras.
COLAB_EXTRAS = [
    "timm==1.0.15",
    "fvcore==0.1.5.post20221221",
    "iopath==0.1.10",
    "yacs==0.1.8",
    "einops==0.8.1",
    "wandb==0.29.0",
    "easydict==1.13",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.constants import CAN_TARGETS
from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils import (
    dataloader_seed_kwargs,
    finish_wandb,
    init_wandb,
    seed_everything,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
PROCESSED_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
WANDB_KEY_PATH = DRIVE_ROOT / "wandb_key.txt"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_OUTPUT_ROOT = Path("/content/stage3_runs")
for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT, LOCAL_PRETRAINED_ROOT, LOCAL_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = REPO / "configs" / "stage3" / "vjepa21b_can_accel_v4a.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
stats = json.loads((MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8"))

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

# Runtime injection keeps the loss tied to the actual split statistics instead
# of duplicating mean/std constants in YAML.
loss_cfg = copy.deepcopy(cfg["loss"])
loss_cfg["normalization"] = {
    name: {
        "mean": float(stats[name]["mean"]),
        "std": float(stats[name]["std"]),
    }
    for name in ("speed_mps", "accel_from_speed_mps2")
}

model_ordinal_thresholds = [
    float(x) for x in cfg["model"].get("accel_ordinal_thresholds_mps2", [])
]
loss_ordinal_thresholds = [
    float(x)
    for x in (
        loss_cfg.get("accel_v3", {})
        .get("ordinal", {})
        .get("thresholds_mps2", [])
    )
]
if model_ordinal_thresholds != loss_ordinal_thresholds:
    raise ValueError(
        "Model/loss ordinal thresholds must match exactly: "
        f"model={model_ordinal_thresholds}, loss={loss_ordinal_thresholds}"
    )
if not model_ordinal_thresholds:
    raise ValueError("v4-A requires non-empty accel ordinal thresholds")

assert_dacon_metric_contract()

print("Repository     :", REPO)
print("Branch / commit:", BRANCH, "/", GIT_COMMIT)
print("Config         :", CFG_PATH)
print("PROCESSED_ROOT :", PROCESSED_ROOT)
print("torch          :", torch.__version__)
print("cuda           :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            :", torch.cuda.get_device_name(0))
print("DACON metric contract: PASS (metrics.py unchanged)")
print("loss mode      :", loss_cfg["mode"])
print("accel stats    :", loss_cfg["normalization"]["accel_from_speed_mps2"])
fusion_cfg = dict(cfg["model"].get("accel_fusion") or {})
if not bool(fusion_cfg.get("enabled", False)):
    raise ValueError("v4-A requires model.accel_fusion.enabled=true")

print("ordinal thr    :", model_ordinal_thresholds)
print("fusion cfg     :", fusion_cfg)


Mounted at /content/drive
Repository     : /content/Blackbox-Detection
Branch / commit: stage3-sangchun / 5a10100
Config         : /content/Blackbox-Detection/configs/stage3/vjepa21b_can_accel_v4a.yaml
PROCESSED_ROOT : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/processed/v1
torch          : 2.8.0+cu126
cuda           : True
gpu            : NVIDIA L4
DACON metric contract: PASS (metrics.py unchanged)
loss mode      : accel_v4_fusion
accel stats    : {'mean': 0.0008632693445153883, 'std': 0.4973502921788985}
ordinal thr    : [0.1, 0.2, 0.3, 0.5]
fusion cfg     : {'enabled': True, 'hidden': 64, 'gate_init': 0.1, 'detach_ordinal_inputs': True}


In [2]:
def _is_usable_file(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= int(min_bytes)
    except OSError:
        return False


def copy_file_to_local(source: Path, destination: Path, *, min_bytes: int = 1, retries: int = 2) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 2):
        tmp = destination.with_name(destination.name + ".copy.tmp")
        try:
            tmp.unlink(missing_ok=True)
            with source.open("rb") as src, tmp.open("wb") as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
            if tmp.stat().st_size < int(min_bytes):
                raise OSError(f"staged file is too small: {tmp.stat().st_size} bytes")
            os.replace(tmp, destination)
            return True
        except OSError as exc:
            last_error = exc
            tmp.unlink(missing_ok=True)
            print(f"copy attempt {attempt} failed:", repr(exc))
            if attempt <= retries:
                time.sleep(2 * attempt)
    print("Drive -> local copy failed:", repr(last_error))
    return False


VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"
if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/facebookresearch/vjepa2.git", str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT], check=True)
ACTUAL_VJEPA_COMMIT = subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

CKPT_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_CKPT_DRIVE = PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT = LOCAL_PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT_URL = "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt"
MIN_VJEPA_BYTES = 1_000_000_000

if not _is_usable_file(VJEPA_CKPT, MIN_VJEPA_BYTES):
    copied = copy_file_to_local(
        VJEPA_CKPT_DRIVE,
        VJEPA_CKPT,
        min_bytes=MIN_VJEPA_BYTES,
    )
    if not copied:
        print("Downloading V-JEPA checkpoint directly to local Colab disk...")
        tmp = VJEPA_CKPT.with_name(VJEPA_CKPT.name + ".download.tmp")
        tmp.unlink(missing_ok=True)
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", str(tmp), VJEPA_CKPT_URL],
            check=True,
        )
        if tmp.stat().st_size < MIN_VJEPA_BYTES:
            raise RuntimeError(f"Downloaded checkpoint is too small: {tmp.stat().st_size} bytes")
        os.replace(tmp, VJEPA_CKPT)

print("V-JEPA commit            :", ACTUAL_VJEPA_COMMIT)
print("V-JEPA checkpoint (LOCAL):", VJEPA_CKPT)
print("checkpoint size MiB      :", f"{VJEPA_CKPT.stat().st_size / 2**20:.1f}")


V-JEPA commit            : 45d025f636dfc58fc2426905fc4a1ab755b1c3e5
V-JEPA checkpoint (LOCAL): /content/pretrained/vjepa2_1_vitb_dist_vitG_384.pt
checkpoint size MiB      : 1587.1


In [3]:
from torch.utils.data import DataLoader
from blackbox_detection.stage3.dataset import Stage3CANDataset

dc = cfg["data"]
tc = cfg["training"]
vc = cfg.get("validation", {})

train_ds = Stage3CANDataset(
    MANIFEST_ROOT / "comma_train.csv",
    PROCESSED_ROOT,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=dc["train_random_flip"],
    max_windows=dc["max_train_windows"],
    seed=SEED,
)

val_ds = Stage3CANDataset(
    MANIFEST_ROOT / "comma_val_id.csv",
    PROCESSED_ROOT,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=False,
    max_windows=dc["max_val_windows"],
    seed=SEED + 1,
)

train_loader = DataLoader(
    train_ds,
    batch_size=tc["batch_size"],
    shuffle=True,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED),
)
val_loader = DataLoader(
    val_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED + 1),
)

print("windows             :", len(train_ds), len(val_ds))
print("batch size          :", tc["batch_size"])
print("grad accumulation   :", tc["grad_accum_steps"])
print("effective batch size:", tc["batch_size"] * tc["grad_accum_steps"])
print("num workers         :", dc["num_workers"])


windows             : 12000 4000
batch size          : 2
grad accumulation   : 4
effective batch size: 8
num workers         : 2


In [4]:
import importlib

from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.models import VJEPA21DenseCAN
from blackbox_detection.utils.checkpoint import load_checkpoint
import blackbox_detection.stage3.trainer as trainer_module

trainer_module = importlib.reload(trainer_module)
CANTrainer = trainer_module.CANTrainer
build_scheduler = trainer_module.build_scheduler

mc = cfg["model"]
fusion_cfg = dict(mc.get("accel_fusion") or {})

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_CKPT,
    num_frames=dc["clip_len"],
    out_layers=tuple(mc["out_layers"]),
    freeze=mc["freeze_backbone"],
)
model = VJEPA21DenseCAN(
    backbone,
    freeze_backbone=mc["freeze_backbone"],
    feature_dim=mc["feature_dim"],
    temporal_hidden=mc["temporal_hidden"],
    temporal_layers=mc["temporal_layers"],
    accel_ordinal_thresholds_mps2=mc.get(
        "accel_ordinal_thresholds_mps2",
        [],
    ),
    accel_fusion_enabled=bool(fusion_cfg.get("enabled", True)),
    accel_fusion_hidden=int(fusion_cfg.get("hidden", 64)),
    accel_fusion_gate_init=float(fusion_cfg.get("gate_init", 0.10)),
    accel_fusion_detach_ordinal_inputs=bool(
        fusion_cfg.get("detach_ordinal_inputs", True)
    ),
)

RUN_VARIANT = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT
LOCAL_RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_VARIANT
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

# First stage this run's own checkpoint, if one exists.
for filename in ("latest.pt", "best.pt"):
    persistent = RUN_DIR / filename
    local = LOCAL_RUN_DIR / filename
    if not local.is_file() and _is_usable_file(persistent):
        ok = copy_file_to_local(persistent, local, min_bytes=1, retries=2)
        print(f"staged v4-A {filename}:", ok, "->", local if ok else None)

V4_RESUME_PATH = LOCAL_RUN_DIR / "latest.pt"
V4_RESUME_AVAILABLE = V4_RESUME_PATH.is_file()

# Fresh v4-A runs warm-start model weights from v3-A best.pt.
warm_cfg = dict(cfg.get("warm_start") or {})
V3_RUN_NAME = str(
    warm_cfg.get("run_name", "vjepa21b_can_v3a_ordinal_frozen")
)
V3_CKPT_NAME = str(warm_cfg.get("checkpoint", "best.pt"))
V3_BEST_DRIVE = OUTPUT_ROOT / V3_RUN_NAME / V3_CKPT_NAME
V3_BEST_LOCAL = (
    LOCAL_PRETRAINED_ROOT
    / f"{V3_RUN_NAME}__{V3_CKPT_NAME}"
)

warm_start_metadata = None
warm_start_applied = False

if not V4_RESUME_AVAILABLE:
    if not _is_usable_file(V3_BEST_LOCAL):
        if not _is_usable_file(V3_BEST_DRIVE):
            raise FileNotFoundError(
                "v3-A warm-start checkpoint not found: "
                f"{V3_BEST_DRIVE}"
            )
        ok = copy_file_to_local(
            V3_BEST_DRIVE,
            V3_BEST_LOCAL,
            min_bytes=1_000_000,
            retries=2,
        )
        if not ok:
            raise OSError(
                "Could not stage v3-A best.pt to local Colab disk"
            )

    warm_start_metadata = load_checkpoint(
        V3_BEST_LOCAL,
        model=model,
        optimizer=None,
        scheduler=None,
        map_location="cpu",
        strict=False,
        restore_rng_state=False,
    )

    expected_new_keys = {
        key
        for key in model.state_dict()
        if "accel_fusion" in key
    }
    missing = set(warm_start_metadata.get("missing_keys") or [])
    unexpected = set(warm_start_metadata.get("unexpected_keys") or [])

    # strict_old_weights=true means every pre-existing v3-A key must load and
    # only the newly introduced fusion parameters may be missing.
    if bool(warm_cfg.get("strict_old_weights", True)):
        if missing != expected_new_keys or unexpected:
            raise RuntimeError(
                "Unexpected v3-A -> v4-A warm-start mismatch. "
                f"missing={sorted(missing)}, "
                f"expected_new={sorted(expected_new_keys)}, "
                f"unexpected={sorted(unexpected)}"
            )

    warm_start_applied = True
    print("v3-A warm start : APPLIED")
    print("source checkpoint:", V3_BEST_LOCAL)
    print("source epoch     :", warm_start_metadata.get("epoch"))
    print("new fusion keys  :", sorted(expected_new_keys))
else:
    print("v4-A latest.pt found: v3-A warm start skipped; full v4-A resume will be used")

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())
fusion_params = sum(
    p.numel()
    for name, p in model.named_parameters()
    if p.requires_grad and "accel_fusion" in name
)
print("trainable params:", trainable_params / 1e6, "M")
print("fusion params   :", fusion_params / 1e3, "K")
print("total params    :", total_params / 1e6, "M")

# Discriminative LR: preserve the learned v3-A temporal/ordinal solution with
# a lower continuation LR while allowing the new zero-initialized fusion path
# to learn quickly.
base_lr = float(tc["learning_rate"])
fusion_lr = float(tc.get("fusion_learning_rate", base_lr))
weight_decay = float(tc["weight_decay"])

buckets = {
    "base_decay": [],
    "base_no_decay": [],
    "fusion_decay": [],
    "fusion_no_decay": [],
}
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    is_fusion = "accel_fusion" in name
    no_decay = parameter.ndim <= 1 or name.endswith(".bias")
    family = "fusion" if is_fusion else "base"
    decay = "no_decay" if no_decay else "decay"
    buckets[f"{family}_{decay}"].append(parameter)

optimizer_groups = []
for group_name in (
    "base_decay",
    "base_no_decay",
    "fusion_decay",
    "fusion_no_decay",
):
    params = buckets[group_name]
    if not params:
        continue
    is_fusion = group_name.startswith("fusion")
    no_decay = group_name.endswith("no_decay")
    optimizer_groups.append(
        {
            "params": params,
            "lr": fusion_lr if is_fusion else base_lr,
            "weight_decay": 0.0 if no_decay else weight_decay,
            "name": group_name,
        }
    )

optimizer = torch.optim.AdamW(optimizer_groups)

micro_steps_per_epoch = min(
    len(train_loader),
    int(tc["max_steps_per_epoch"]),
)
optimizer_steps_per_epoch = max(
    math.ceil(
        micro_steps_per_epoch / int(tc["grad_accum_steps"])
    ),
    1,
)
total_optimizer_steps = (
    optimizer_steps_per_epoch * int(tc["epochs"])
)

scheduler = build_scheduler(
    optimizer,
    total_steps=total_optimizer_steps,
    warmup_ratio=tc["warmup_ratio"],
    min_ratio=tc["min_learning_rate_ratio"],
)

print("base LR                :", base_lr)
print("fusion LR              :", fusion_lr)
print("micro steps / epoch    :", micro_steps_per_epoch)
print("optimizer steps / epoch:", optimizer_steps_per_epoch)
print("total optimizer steps  :", total_optimizer_steps)
for group in optimizer.param_groups:
    print(
        "optimizer group:",
        group.get("name"),
        "lr=", group["lr"],
        "wd=", group["weight_decay"],
        "params=", sum(p.numel() for p in group["params"]),
    )

lc = cfg.get("logging", {})
WANDB_ENABLED = bool(lc.get("wandb_enabled", True))
wandb_run = None

if WANDB_ENABLED:
    import wandb
    if not WANDB_KEY_PATH.is_file():
        raise FileNotFoundError(
            f"W&B key file not found: {WANDB_KEY_PATH}"
        )
    wandb_key = WANDB_KEY_PATH.read_text(
        encoding="utf-8"
    ).strip()
    if not wandb_key:
        raise ValueError(
            f"W&B key file is empty: {WANDB_KEY_PATH}"
        )
    wandb.login(key=wandb_key, relogin=False)
    del wandb_key
    finish_wandb()

    run_id_path = RUN_DIR / "wandb_run_id.txt"
    stored_run_id = (
        run_id_path.read_text(encoding="utf-8").strip()
        if run_id_path.is_file()
        else None
    )
    stored_run_id = stored_run_id or None

    run_name = (
        f"{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}"
    )
    wandb_run = init_wandb(
        project=str(
            lc.get("wandb_project", "blackbox-stage3")
        ),
        entity=os.getenv("WANDB_ENTITY") or None,
        name=run_name,
        group=str(
            lc.get(
                "wandb_group",
                "vjepa21b_can_accel_v4a",
            )
        ),
        tags=[
            "stage3",
            "vjepa2",
            "comma2k19",
            "continuous-can",
            "accel-v4a",
            "ordinal-fusion",
            "v3a-warm-start",
            "frozen-backbone",
        ],
        run_id=stored_run_id,
        resume="allow",
        config={
            "git_commit": GIT_COMMIT,
            "vjepa_commit": ACTUAL_VJEPA_COMMIT,
            "run_variant": RUN_VARIANT,
            "seed": SEED,
            "num_train_windows": len(train_ds),
            "num_val_windows": len(val_ds),
            "trainable_params": trainable_params,
            "fusion_params": fusion_params,
            "total_params": total_params,
            "effective_batch_size": (
                tc["batch_size"]
                * tc["grad_accum_steps"]
            ),
            "warm_start_applied": warm_start_applied,
            "warm_start_run": V3_RUN_NAME,
            "warm_start_checkpoint": V3_CKPT_NAME,
            "config": cfg,
            "runtime_loss_config": loss_cfg,
        },
        directory=LOCAL_RUN_DIR / "wandb",
        mode=os.getenv("WANDB_MODE") or None,
    )
    if stored_run_id is None:
        run_id_path.write_text(
            wandb_run.id,
            encoding="utf-8",
        )
    print("W&B run:", wandb_run.name)
    print("W&B id :", wandb_run.id)
    print("W&B url:", wandb_run.url)

trainer_config = {
    "git_commit": GIT_COMMIT,
    "vjepa_commit": ACTUAL_VJEPA_COMMIT,
    "run_variant": RUN_VARIANT,
    "seed": SEED,
    "data": dc,
    "model": mc,
    "training": tc,
    "warm_start": warm_cfg,
    "loss": loss_cfg,
    "validation": vc,
    "logging": lc,
    "warm_start_from_v3a": True,
    "warm_start_applied_this_runtime": warm_start_applied,
    "fresh_optimizer_from_v3a": True,
    "ordinal_thresholds_mps2": model_ordinal_thresholds,
}

trainer = CANTrainer(
    model,
    optimizer,
    scheduler=scheduler,
    grad_accum_steps=tc["grad_accum_steps"],
    grad_clip_norm=tc["grad_clip_norm"],
    amp_dtype=tc["amp_dtype"],
    loss_weights=loss_cfg,
    stats=stats,
    proxy_rules=vc.get("proxy_rules", {}),
    output_dir=LOCAL_RUN_DIR,
    sync_dir=RUN_DIR,
    wandb_enabled=WANDB_ENABLED,
    log_interval=tc.get("log_interval", 20),
    config=trainer_config,
)

print("local run dir     :", LOCAL_RUN_DIR)
print("persistent run dir:", RUN_DIR)


staged v4-A latest.pt: True -> /content/stage3_runs/vjepa21b_can_v4a_ordinal_fusion/latest.pt
staged v4-A best.pt: True -> /content/stage3_runs/vjepa21b_can_v4a_ordinal_fusion/best.pt
v4-A latest.pt found: v3-A warm start skipped; full v4-A resume will be used
trainable params: 3.061902 M
fusion params   : 1.026 K
total params    : 89.895054 M
base LR                : 5e-05
fusion LR              : 0.0002
micro steps / epoch    : 2000
optimizer steps / epoch: 500
total optimizer steps  : 1500
optimizer group: base_decay lr= 6.666666666666667e-07 wd= 0.01 params= 3053568
optimizer group: base_no_decay lr= 6.666666666666667e-07 wd= 0.0 params= 7308
optimizer group: fusion_decay lr= 2.666666666666667e-06 wd= 0.01 params= 832
optimizer group: fusion_no_decay lr= 2.666666666666667e-06 wd= 0.0 params= 194


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sangchun1 (sangchun1-chung-ang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


W&B run: vjepa21b_can_v4a_ordinal_fusion__seed20260918__5a10100
W&B id : 8fs81pmp
W&B url: https://wandb.ai/sangchun1-chung-ang-university/blackbox-stage3/runs/8fs81pmp
local run dir     : /content/stage3_runs/vjepa21b_can_v4a_ordinal_fusion
persistent run dir: /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v4a_ordinal_fusion


In [5]:
# One-batch v4-A forward/loss/backward smoke before spending an epoch.
from blackbox_detection.stage3.losses import can_multitask_loss

smoke = next(iter(train_loader))
video = smoke["video"][:1].to(trainer.device)
target = smoke["target"][:1].to(trainer.device)
valid = smoke["valid"][:1].to(trainer.device)

optimizer.zero_grad(set_to_none=True)
model.train()
with torch.autocast(
    device_type=trainer.device.type,
    dtype=trainer.amp_dtype,
    enabled=trainer.device.type == "cuda",
):
    smoke_out = model(video)
    smoke_loss, smoke_parts = can_multitask_loss(
        smoke_out,
        target,
        valid,
        loss_cfg,
    )

if not torch.isfinite(smoke_loss):
    raise RuntimeError(
        f"Non-finite v4-A smoke loss: {smoke_loss}"
    )

expected_shape = (
    1,
    dc["clip_len"],
    len(model_ordinal_thresholds),
    2,
)
actual_shape = tuple(
    smoke_out["accel_ordinal_logits"].shape
)
if actual_shape != expected_shape:
    raise RuntimeError(
        "Unexpected ordinal output shape: "
        f"{actual_shape}, expected {expected_shape}"
    )

required_fusion_outputs = {
    "accel_raw_from_speed_mps2",
    "accel_fusion_residual",
    "accel_fusion_gate",
    "accel_ordinal_signed_score",
}
missing_outputs = required_fusion_outputs - set(smoke_out)
if missing_outputs:
    raise RuntimeError(
        f"Missing v4-A fusion outputs: {sorted(missing_outputs)}"
    )

raw_accel = smoke_out["accel_raw_from_speed_mps2"]
fused_accel = smoke_out["accel_from_speed_mps2"]
initial_delta_max = float(
    (fused_accel - raw_accel)
    .abs()
    .max()
    .detach()
    .cpu()
)

# A fresh v3-A warm-start must be functionally identical before the first
# optimizer step because the fusion residual's last layer is zero-initialized.
if warm_start_applied and initial_delta_max > 1e-7:
    raise RuntimeError(
        "Fresh v4-A warm-start changed scalar acceleration "
        f"before training: max |fused-raw|={initial_delta_max}"
    )

smoke_loss.backward()

fusion_last = model.head.accel_fusion_mlp[-1]
fusion_grad = fusion_last.weight.grad
if (
    fusion_grad is None
    or not torch.isfinite(fusion_grad).all()
):
    raise RuntimeError(
        "Fusion output layer did not receive a finite gradient"
    )
fusion_grad_norm = float(
    fusion_grad.float().norm().detach().cpu()
)
if fusion_grad_norm <= 0:
    raise RuntimeError(
        "Fusion output-layer gradient norm is zero"
    )

ordinal_grad = model.head.accel_ordinal_head.weight.grad
if (
    ordinal_grad is None
    or not torch.isfinite(ordinal_grad).all()
):
    raise RuntimeError(
        "Ordinal head did not receive a finite gradient"
    )
ordinal_grad_norm = float(
    ordinal_grad.float().norm().detach().cpu()
)

print("v4-A smoke total       :", float(smoke_loss.detach().cpu()))
print("ordinal output shape   :", actual_shape)
print("initial |fused-raw|max :", initial_delta_max)
print(
    "fusion gate            :",
    float(
        smoke_out["accel_fusion_gate"]
        .detach()
        .float()
        .cpu()
    ),
)
print("fusion grad norm       :", fusion_grad_norm)
print("ordinal grad norm      :", ordinal_grad_norm)
print(json.dumps(smoke_parts, indent=2))

optimizer.zero_grad(set_to_none=True)
del (
    video,
    target,
    valid,
    smoke_out,
    smoke_loss,
    fusion_grad,
    ordinal_grad,
)
if torch.cuda.is_available():
    torch.cuda.empty_cache()


v4-A smoke total       : 11.664885520935059
ordinal output shape   : (1, 16, 4, 2)
initial |fused-raw|max : 0.0
fusion gate            : 0.10009765625
fusion grad norm       : 1.8430101871490479
ordinal grad norm      : 0.9591602087020874
{
  "speed_mps": 0.7885536551475525,
  "accel_from_speed_mps2": 1.8742536306381226,
  "steering_deg": 2.5147743225097656,
  "yaw_rate_rps": 4.183076858520508,
  "accel_v2/weighted_regression": 1.8742533922195435,
  "accel_v2/mean_sample_weight": 3.0,
  "accel_v2/multi_threshold_margin": 0.7075397968292236,
  "accel_v2/speed_delta": 0.0506451316177845,
  "accel_v2/speed_accel_consistency": 0.8528686761856079,
  "accel_v3/ordinal_bce": 0.6782071590423584,
  "accel_v3/ordinal_monotonic": 0.07421875,
  "accel_v3/ordinal_total": 0.6856229305267334,
  "accel_v3/decel_bce_thr_0p10": 0.6009734869003296,
  "accel_v3/accel_bce_thr_0p10": 0.5496346950531006,
  "accel_v3/decel_bce_thr_0p20": 0.7639117240905762,
  "accel_v3/accel_bce_thr_0p20": 0.7606084942817688,

In [6]:
resume_path = LOCAL_RUN_DIR / "latest.pt"
print(
    "resume:",
    resume_path if resume_path.is_file() else None,
)
if resume_path.is_file():
    print(
        "NOTE: v4-A latest.pt takes precedence over "
        "the one-time v3-A warm start."
    )

history = trainer.fit(
    train_loader,
    val_loader,
    epochs=tc["epochs"],
    max_train_steps=tc["max_steps_per_epoch"],
    max_val_steps=tc.get("max_val_steps", 500),
    resume_from=(
        resume_path if resume_path.is_file() else None
    ),
    early_stopping_patience=tc.get(
        "early_stopping_patience",
        0,
    ),
    backfill_validation_on_resume=bool(
        vc.get("backfill_on_resume", True)
    ),
)

history_df = pd.DataFrame(
    [
        {
            "epoch": x["epoch"],
            "minutes": x["minutes"],
            "learning_rate": x.get("learning_rate"),
            "global_step": x.get("global_step"),
            "max_gpu_memory_gib": x.get(
                "max_gpu_memory_gib"
            ),
            **{
                f"train/{k}": v
                for k, v in x["train"].items()
            },
            **{
                f"val/{k}": v
                for k, v in x["val"].items()
            },
        }
        for x in history
    ]
)

display(history_df)


resume: /content/stage3_runs/vjepa21b_can_v4a_ordinal_fusion/latest.pt
NOTE: v4-A latest.pt takes precedence over the one-time v3-A warm start.
[2026-09-23 17:46:01] INFO | blackbox_detection.stage3.can | Resumed from /content/stage3_runs/vjepa21b_can_v4a_ordinal_fusion/latest.pt at epoch 2 (next=3, best val total=1.602302, global_step=1000).


Train 3:   0%|          | 0/2000 [00:00<?, ?it/s]

Val 3:   0%|          | 0/500 [00:00<?, ?it/s]

[2026-09-23 18:18:39] INFO | blackbox_detection.stage3.can | Epoch 3/3 | train 1.563261 | val 1.586531 | proxy-F1(mean) 0.5716 | lr 2.500e-06 | 32.6 min | peak 0.69 GiB <- best CAN-pretrain


,epoch,minutes,learning_rate,global_step,max_gpu_memory_gib,train/accel_from_speed_mps2,train/accel_v2/mean_sample_weight,train/accel_v2/multi_threshold_margin,train/accel_v2/speed_accel_consistency,train/accel_v2/speed_delta,...,val/proxy/conservative/f1_accel_CONSTANT,val/proxy/conservative/f1_accel_STOPPED,val/proxy/conservative/f1_steer_LEFT,val/proxy/conservative/f1_steer_STRAIGHT,val/proxy/conservative/f1_steer_RIGHT,val/proxy/robust_mean_stage3_score,val/proxy/robust_min_stage3_score,val/proxy/robust_max_stage3_score,val/proxy/robust_mean_accel_macro_f1,val/proxy/robust_mean_steer_macro_f1
0,1,65.733267,0.000040,500,0.692959,0.360343,1.959714,0.330648,0.157154,0.004343,...,0.828420,0.941721,0.500935,0.985162,0.574924,0.571775,0.557051,0.599371,0.554265,0.612630
1,2,29.789435,0.000016,1000,0.693875,0.334673,1.959714,0.307153,0.146310,0.004116,...,0.825322,0.928551,0.529511,0.983336,0.547804,0.570157,0.539326,0.609084,0.549642,0.618027
2,3,32.556994,0.000003,1500,0.693875,0.315467,1.959714,0.289576,0.138483,0.004041,...,0.826491,0.930246,0.540541,0.982870,0.547315,0.571552,0.531785,0.618398,0.550595,0.620451


In [7]:
if len(history_df):
    best_idx = (
        history_df["val/total"]
        .astype(float)
        .idxmin()
    )
    best_row = history_df.loc[best_idx]

    summary = {
        "run_variant": RUN_VARIANT,
        "git_commit": GIT_COMMIT,
        "vjepa_commit": ACTUAL_VJEPA_COMMIT,
        "best_epoch": int(best_row["epoch"]),
        "best_val_total": float(
            best_row["val/total"]
        ),
        "checkpoint_selection": (
            "minimum v4-A val/total; proxy and "
            "ordinal metrics remain diagnostic"
        ),
        "proxy_metric_status": (
            "diagnostic only; not official DACON "
            "labels/thresholds"
        ),
        "ordinal_metric_status": (
            "training/diagnostic only; not DACON "
            "labels/thresholds"
        ),
        "warm_start_from_v3a": True,
        "warm_start_run": V3_RUN_NAME,
        "warm_start_checkpoint": V3_CKPT_NAME,
        "fresh_optimizer_from_v3a": True,
        "ordinal_thresholds_mps2": (
            model_ordinal_thresholds
        ),
        "fusion_detach_ordinal_inputs": bool(
            fusion_cfg.get(
                "detach_ordinal_inputs",
                True,
            )
        ),
        "base_learning_rate": base_lr,
        "fusion_learning_rate": fusion_lr,
        "num_train_windows": int(len(train_ds)),
        "num_val_windows": int(len(val_ds)),
        "trainable_params": int(trainable_params),
        "fusion_params": int(fusion_params),
    }

    if warm_start_metadata is not None:
        source_epoch = warm_start_metadata.get("epoch")
        if source_epoch is not None:
            summary["warm_start_source_epoch"] = int(
                source_epoch
            )
        source_best = warm_start_metadata.get(
            "best_score"
        )
        if source_best is not None:
            summary[
                "warm_start_source_best_score"
            ] = float(source_best)

    for column, value in best_row.items():
        if (
            isinstance(column, str)
            and column.startswith("val/")
        ):
            try:
                summary[f"best_{column}"] = float(
                    value
                )
            except (TypeError, ValueError):
                pass

    proxy_col = (
        "val/proxy/robust_mean_stage3_score"
    )
    if proxy_col in history_df.columns:
        s = pd.to_numeric(
            history_df[proxy_col],
            errors="coerce",
        )
        if s.notna().any():
            i = s.idxmax()
            summary[
                "diagnostic_best_proxy_epoch"
            ] = int(history_df.loc[i, "epoch"])
            summary[
                "diagnostic_best_proxy_mean_stage3_score"
            ] = float(s.loc[i])

    # Persist a compact v3-A baseline next to the v4-A result.
    v3_summary_path = (
        OUTPUT_ROOT
        / V3_RUN_NAME
        / "summary.json"
    )
    if v3_summary_path.is_file():
        try:
            v3_summary = json.loads(
                v3_summary_path.read_text(
                    encoding="utf-8"
                )
            )
            v3_keys = [
                "best_val/diag/accel/pred_to_gt_std_ratio",
                "best_val/diag/accel/pred_vs_gt_slope",
                "best_val/diag/accel/correlation",
                "best_val/proxy/medium/dynamic_to_constant_rate",
                "best_val/proxy/medium/f1_accel_ACCELERATING",
                "best_val/proxy/medium/f1_accel_DECELERATING",
                "best_val/proxy/robust_mean_accel_macro_f1",
                "best_val/proxy/robust_mean_stage3_score",
                "best_val/aux/ordinal/mean_accel_auc",
                "best_val/aux/ordinal/mean_decel_auc",
            ]
            for key in v3_keys:
                if key in v3_summary:
                    summary[
                        f"v3a_{key}"
                    ] = float(v3_summary[key])
        except Exception as exc:
            print(
                "v3-A summary comparison warning:",
                repr(exc),
            )

    local_summary = (
        LOCAL_RUN_DIR / "summary.json"
    )
    local_summary.write_text(
        json.dumps(
            summary,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    trainer._sync_file(local_summary)

    print(json.dumps(summary, indent=2))

    diag_cols = [
        # Fused scalar acceleration.
        "val/diag/accel/pred_to_gt_std_ratio",
        "val/diag/accel/pred_vs_gt_slope",
        "val/diag/accel/correlation",
        # Raw pre-fusion scalar acceleration.
        "val/diag/accel_raw/pred_to_gt_std_ratio",
        "val/diag/accel_raw/pred_vs_gt_slope",
        "val/diag/accel_raw/correlation",
        # Direct fusion effect.
        "val/diag/accel_fusion/fused_minus_raw_abs_mean_mps2",
        "val/diag/accel_fusion/fused_minus_raw_std_mps2",
        # Class-consistency diagnostics.
        "val/proxy/medium/dynamic_to_constant_rate",
        "val/proxy/medium/f1_accel_ACCELERATING",
        "val/proxy/medium/f1_accel_DECELERATING",
        "val/proxy/robust_mean_accel_macro_f1",
        "val/proxy/robust_mean_stage3_score",
        # Ordinal branch should remain healthy.
        "val/aux/ordinal/mean_accel_auc",
        "val/aux/ordinal/mean_decel_auc",
    ]
    present = [
        c
        for c in diag_cols
        if c in history_df.columns
    ]
    if present:
        print(
            "\nV4-A fused/raw acceleration diagnostics by epoch:"
        )
        display(
            history_df[
                ["epoch", *present]
            ]
        )

    if (
        WANDB_ENABLED
        and wandb_run is not None
    ):
        for key, value in summary.items():
            if (
                isinstance(
                    value,
                    (str, int, float, bool),
                )
                or value is None
            ):
                wandb_run.summary[key] = value
        try:
            for filename in (
                "summary.json",
                "history.csv",
                "train_config.json",
                "train.log",
            ):
                path = LOCAL_RUN_DIR / filename
                if path.is_file():
                    wandb_run.save(
                        str(path),
                        base_path=str(
                            LOCAL_RUN_DIR
                        ),
                    )
        except Exception as exc:
            print(
                "W&B file upload warning:",
                repr(exc),
            )

if WANDB_ENABLED:
    finish_wandb()
    print("W&B run finished.")

print(
    "local latest     :",
    LOCAL_RUN_DIR / "latest.pt",
)
print(
    "persistent latest:",
    RUN_DIR / "latest.pt",
)
print(
    "persistent best  :",
    RUN_DIR / "best.pt",
)


{
  "run_variant": "vjepa21b_can_v4a_ordinal_fusion",
  "git_commit": "5a10100",
  "vjepa_commit": "45d025f636dfc58fc2426905fc4a1ab755b1c3e5",
  "best_epoch": 3,
  "best_val_total": 1.5865314546525477,
  "checkpoint_selection": "minimum v4-A val/total; proxy and ordinal metrics remain diagnostic",
  "proxy_metric_status": "diagnostic only; not official DACON labels/thresholds",
  "ordinal_metric_status": "training/diagnostic only; not DACON labels/thresholds",
  "warm_start_from_v3a": true,
  "warm_start_run": "vjepa21b_can_v3a_ordinal_frozen",
  "warm_start_checkpoint": "best.pt",
  "fresh_optimizer_from_v3a": true,
  "ordinal_thresholds_mps2": [
    0.1,
    0.2,
    0.3,
    0.5
  ],
  "fusion_detach_ordinal_inputs": true,
  "base_learning_rate": 5e-05,
  "fusion_learning_rate": 0.0002,
  "num_train_windows": 12000,
  "num_val_windows": 4000,
  "trainable_params": 3061902,
  "fusion_params": 1026,
  "best_val/accel_from_speed_mps2": 0.3532830063691363,
  "best_val/accel_v2/mean_samp

,epoch,val/diag/accel/pred_to_gt_std_ratio,val/diag/accel/pred_vs_gt_slope,val/diag/accel/correlation,val/diag/accel_raw/pred_to_gt_std_ratio,val/diag/accel_raw/pred_vs_gt_slope,val/diag/accel_raw/correlation,val/diag/accel_fusion/fused_minus_raw_abs_mean_mps2,val/diag/accel_fusion/fused_minus_raw_std_mps2,val/proxy/medium/dynamic_to_constant_rate,val/proxy/medium/f1_accel_ACCELERATING,val/proxy/medium/f1_accel_DECELERATING,val/proxy/robust_mean_accel_macro_f1,val/proxy/robust_mean_stage3_score,val/aux/ordinal/mean_accel_auc,val/aux/ordinal/mean_decel_auc
0,1,0.388760,0.163997,0.421847,0.385950,0.162868,0.421991,0.001024,0.001426,0.755530,0.316389,0.232889,0.554265,0.571775,0.748704,0.727299
1,2,0.438253,0.205804,0.469600,0.433336,0.203601,0.469846,0.001755,0.002567,0.725602,0.340261,0.280924,0.549642,0.570157,0.750307,0.736449
2,3,0.462504,0.226435,0.489585,0.456562,0.223697,0.489959,0.002259,0.003178,0.701529,0.341036,0.320886,0.550595,0.571552,0.751210,0.739292


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
optim/learning_rate,█▇▇▆▆▅▅▅▄▄▄▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
selection/best_val_total,▁
system/epoch_minutes,▁
system/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇█
system/max_gpu_memory_gib,▁
train/accel_from_speed_mps2,▁
train/accel_v2/mean_sample_weight,▁
train/accel_v2/multi_threshold_margin,▁
train/accel_v2/speed_accel_consistency,▁
+180,...


W&B run finished.
local latest     : /content/stage3_runs/vjepa21b_can_v4a_ordinal_fusion/latest.pt
persistent latest: /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v4a_ordinal_fusion/latest.pt
persistent best  : /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v4a_ordinal_fusion/best.pt


## How to judge v4-A

v4-A tests one narrow hypothesis: **the v3-A ordinal branch already contains useful longitudinal information, but the scalar acceleration head is not converting it into enough signed magnitude.**

The first comparison is therefore **fused vs raw inside the same model**:

- `val/diag/accel/pred_to_gt_std_ratio` — final fused acceleration.
- `val/diag/accel_raw/pred_to_gt_std_ratio` — pre-fusion scalar branch.
- `val/diag/accel/correlation` vs `val/diag/accel_raw/correlation`.
- `val/diag/accel_fusion/fused_minus_raw_abs_mean_mps2`.

Then track the Stage-3-like diagnostics:

- `val/proxy/medium/dynamic_to_constant_rate`
- `val/proxy/medium/f1_accel_ACCELERATING`
- `val/proxy/medium/f1_accel_DECELERATING`
- `val/proxy/robust_mean_accel_macro_f1`

The completed v3-A reference was approximately:

- fused-equivalent scalar std ratio: **0.288**
- scalar correlation: **0.401**
- scalar slope: **0.116**
- medium dynamic→CONSTANT: **0.784**
- medium ACCEL F1: **0.225**
- medium DECEL F1: **0.303**
- robust accel Macro-F1: **0.527**

A useful v4-A should increase scalar amplitude without destroying alignment: ideally fused std ratio rises clearly above the raw branch and v3-A baseline, correlation stays stable or improves, dynamic→CONSTANT falls, and ACCEL/DECEL F1 improve together.

Also keep an eye on:

- `val/aux/ordinal/mean_accel_auc`
- `val/aux/ordinal/mean_decel_auc`

The ordinal inputs are detached from scalar fusion gradients, so those AUCs should remain healthy while the fusion MLP learns to map ordinal evidence into continuous acceleration.

### Decision after v4-A

- **Fusion clearly improves amplitude/class consistency:** keep the v4-A architecture and move on to A2D2/domain-diverse pretraining before touching the backbone.
- **Fusion changes the output but does not improve class consistency:** tune the fusion mapping/gate or derive a more explicit ordinal expected-magnitude estimator.
- **Ordinal AUC remains strong but fusion still cannot repair scalar acceleration:** then test v4-B with the last V-JEPA blocks unfrozen/adapted.
- **Do not extend the old v3-A cosine schedule.** v4-A already uses a fresh continuation optimizer/scheduler.

All `proxy/...` and `aux/ordinal/...` metrics are diagnostics only. `metrics.py` remains unchanged.
